In [114]:
'''
Qs: 
Unstocked Forest Qs:
What is an approporiate amount of time for n_years_unstocked_forest? 
If something starts out as short veg should we consider that Forest or Grassland depending on other circumstances? (i.e. if driver is logging)
- Check short veg with open woodlands in Africa to see if areas that should be forest are not being classified correctly? 

Is the logging concessions data rasterized? What are the raster values (i.e. 0/1)?
- Canada: 2016; 
- Equatorial Guinea: 2013; 
- Indonesia: 2021; 
- Liberia: 2016; 
- Malaysia: 2010; 
- Republic of the Congo: 2013
- Otherwise: unknown


Shifting Cultivation Qs:
Proposed rules said to classify as "Cropland" if mix of short veg/ tall veg and driver == Shifting cultivation. 
- Instead, if it starts out as forest, assume forest until short veg or cropland occurs. Once short veg or cropland occurs, assume cropland for the rest of the timeseries if driver == Shifting cultivation? 
- Otherwise, if it starts out as short veg, assume cropland for the whole time series if driver ==  Shifting cultivation. 
Proposed rule said to classify as "Cropland" + driver == Shifting cultivation. Do we want to extend "Cropland" classification in the timeseries when driver != Shifting cultivation?
Should we use n_years_fallow_cropland to determine transition from cropland back to forest? 


Cautions: 
- No shifting cultivation establishment year. So assuming Cropland the first time tall --> short veg or after "Cropland" LC class.



TODOs: 
How to handle LC -> LU transitions within 5-year intervals if n_years < 5? 
'''

import re
import pandas as pd
import numpy as np

In [115]:
n_years_unstocked_forest = 3    # number of years forest is allowed to be unstocked before being considered a forest --> grassland conversion

years = [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

lc_code_map = {
    0: "bare",
    1: "short veg dry",
    2: "cultivated dry",
    3: "forest dry",
    4: "wetland",
    5: "short veg wet",
    6: "cultivated wet",
    7: "forest wet",
    8: "water",
    9: "snow/ice",
    10: "cropland",
    11: "built up",
}

driver_code_map = {
    1: "Permanent agriculture",
    2: "Hard commodities",
    3: "Shifting cultivation",
    4: "Logging",
    5: "Wildfire",
    6: "Settlements & infrastructure",
    7: "Other natural disturbances"
}

sdpt_code_map = {
    1: "oil palm",
    2: "wood fiber",
    3: "other",
}

In [116]:
# Default LC numeric values -> LU classes 
forest_lc = {3, 7}              # Tall vegetation
grass_lc = {1, 2, 5, 6}         # Short veg
cropland_lc = {10}              # Cropland
settlement_lc = {11}            # Built up
wetland_lc = {4}                # Wetland
other_lc = {0, 8, 9}            # Bare, water, snow/ice

# Lookup table from LC code -> token
lc_token_map = {
    **{v: "F" for v in forest_lc},
    **{v: "G" for v in grass_lc},
    **{v: "C" for v in cropland_lc},
    **{v: "S" for v in settlement_lc},
    **{v: "W" for v in wetland_lc},
    **{v: "O" for v in other_lc},
}

# Function to get land use token per land cover numeric value (tokens used for regex exception rules)
def token_for_lc(v):
    return lc_token_map.get(v, "-")

In [117]:
# Exceptions to the default land cover to land use assignments
#TODO: Update tokens, token_seq with overide values or lust LU?

# If pixel is in a logging concessions area, assume any short vegetation is unstocked forest so land use remains "Forest Land"
# TODO: Should we apply to other LUs, not just short veg? (i.e. cropland, other, wetland, settlements)
def apply_logging_concessions(ctx):
    if ctx["logging_concession"] == 1:
        for i, lc in enumerate(ctx["tokens"]):
            if lc == "G":
                ctx["lu_vals"][i] = "Forest land"

# If pixel is in SDPT planted forest, assume any short vegetation is unstocked forest so land use remains "Forest Land"
# TODO: Should we apply to other LUs, not just short veg? (i.e. cropland, other, wetland, settlements)
def apply_sdpt_planted_forest(ctx):
    if ctx["sdpt"] == 2:
        for i, lc in enumerate(ctx["tokens"]):
            if lc == "G":
                ctx["lu_vals"][i] = "Forest land"
                
# If tall veg switches to short veg for less than n_years_unstocked_forest and returns back to tall veg assume unstocked forest so land use remains "Forest Land"
# TODO: Should we handle terminal exceptions in the main exceptions? 
def apply_temp_unstocked_forest(ctx):
    pattern = rf"F(G{{1,{ctx['n_unstocked']}}})(?=F)"   # Positive look ahead for following F so it's not included in match (i.e. FGFGFGFGF)
    for m in re.finditer(pattern, ctx["token_seq"]):
        if m:
            a, b = m.span(1)
            for i in range(a, b):
                ctx["lu_vals"][i] = "Forest land"

# If tall veg switches to short veg at the end of the timeseries for less tha n_years_unstocked_forest, assume unstocked forest so land use remains "Forest Land"
# TODO: Also apply to initial Gs
def apply_temp_unstocked_forest_terminal(ctx):
    pattern = rf"F(G{{1,{ctx['n_unstocked']}}})$"
    m = re.search(pattern, ctx["token_seq"])
    if m:
        a, b = m.span(1)
        for i in range(a, b):
            ctx["lu_vals"][i] = "Forest land"
    return

# If timeseries end with short veg after tall veg and driver is "logging" assume unstocked forest so land use remains "Forest Land"
# Note: short veg not limited to n_years_unstocked_forest here
# TODO: Should we also apply logging driver to any interior G values as well like SDPT or logging concession? 
def apply_forest_logging_terminal(ctx):
    if ctx["driver"] == 4:
        pattern = r"F(G+)$"
        m = re.search(pattern, ctx["token_seq"])
        if m:
            a, b = m.span(1) 
            for i in range(a, b):
                ctx["lu_vals"][i] = "Forest land"





In [118]:
#TODO: If only allowing for one rule to apply, then need to call terminal funcitons in other exceptions (i.e unstocked forest rule also checks unstocked forest at the end even if LC doesnt return to F, etc)
rules = {"kind": 
        # Default rules to go from land cover to land use have already been applied (via tokens)
        # so just assigning the labels and names for those default assumptions here
        {"default": {
            
                    "F": {"label": "Forest land",
                          "names": "forest_default"},
                    "N": {"label": "Grassland",
                          "names": "grass_default"},
                    "G": {"label": "Grassland",
                          "names": "grass_default"},
                    "C": {"label": "Cropland",
                          "names": "cropland_default"},
                    "S": {"label": "Settlements",
                          "names": "settlements_default"},
                    "W": {"label": "Wetlands",
                          "names": "wetlands_default"},
                    "O": {"label": "Other land",
                          "names": "other_default"},
                    },
        
        # Each exception calls a function which uses regex to see if that condition applies to this lu timeseries
        # If that condition applies, the lu values are updated accordingly and the name is updated with the exception applied
        "exception": [
            
            # Confusion between Forest and Grassland exceptions
            {"name": "forest_logging_concessions",
             "apply": apply_logging_concessions},

            {"name": "forest_sdpt_planted_forest",
             "apply": apply_sdpt_planted_forest},

            {"name": "forest_temp_unstocked_forest",
             "apply": apply_temp_unstocked_forest},
            
            {"name": "forest_temp_unstocked_terminal",
             "apply": apply_temp_unstocked_forest_terminal},

            {"name": "forest_logging_terminal",
             "apply": apply_forest_logging_terminal},
            
            # Confusion between Forest and Cropland exceptions

        ]
    }
}

# Function to apply default rules to go from GLAD land cover to IPCC land Use
def lc_to_lu_default(lc_vals, rules_table):
    lu_vals = []
    for lc in lc_vals:
        try:
            lu = lc_token_map[lc]
            assigned = rules_table["kind"]["default"][lu]["label"]
            lu_vals.append(assigned)
        except KeyError:
            lu_vals.append(None)
    
    return lu_vals

In [119]:
# Main function to apply the regex rules to the LC timeseries and returns the final LU timeseries
def classify_scenario(scenario_id, lc_timeseries, rules, driver=None, sdpt=None, logging_concession=None):
    
    # Create array with default LU tokens using LC timeseries
    lc_vals = [int(v) for v in lc_timeseries]
    lu_vals = lc_to_lu_default(lc_vals, rules)
    print(f"lu_vals: {lu_vals}")

    # Build tokens sequence (string with no spaces) and apply exception rules
    tokens = [token_for_lc(v) for v in lc_vals]
    token_seq = "".join(tokens)
    print(f"tokens: {tokens}")
    print(f"token_seq: {token_seq}")

    # Inputs for exception rules
    ctx = {
        "scenario_id": scenario_id,
        "lc_vals": lc_vals,
        "lu_vals": lu_vals,
        "tokens": tokens,
        "token_seq": token_seq,
        "driver": driver,
        "sdpt": sdpt,
        "logging_concession": logging_concession,
        "n_unstocked": n_years_unstocked_forest
    }
    
    # Override defaults if exception(s) apply
    for rule in rules["kind"]["exception"]:
        rule["apply"](ctx)

    return lu_vals

In [120]:
# Iterate through each scenarios and classify LC to create LU timeseries
def classify_dataframe(df, rules, lc_cols):
    out = []
    for idx, row in df.iterrows():
        driver = int(row["driver"]) if pd.notna(row["driver"]) else None
        sdpt   = int(row["sdpt"])   if pd.notna(row["sdpt"])   else None
        logc   = int(row["logging_concession"]) if pd.notna(row["logging_concession"]) else None
        lc_ts  = [row.get(c) for c in lc_cols]

        lu_ts = classify_scenario(row["id"], lc_ts, rules, driver=driver, sdpt=sdpt, logging_concession=logc)

        # Create annual land use columns
        lu_cols = {f"lu_{y}": lu_ts[i] for i,y in enumerate(years)}

        # Create time step transitions and conversion flags
        conversion = False
        trans_cols = {}
        for i,(a,b) in enumerate(zip(years[:-1], years[1:])):
            a_lu, b_lu = lu_ts[i], lu_ts[i+1]
            key = f"{a}_{b}"
            if a_lu == b_lu:
                trans_cols[key] = f"{a_lu} remaining {b_lu}"
            else:
                trans_cols[key] = f"{a_lu} to {b_lu}"
                conversion = True

        out.append({
            "id": row["id"],
            "driver": driver_code_map.get(driver, str(driver) if driver is not None else None),
            "sdpt":   sdpt_code_map.get(sdpt,   str(sdpt)   if sdpt   is not None else None),
            "logging_concession": "yes" if logc == 1 else None,
            **lu_cols,
            **trans_cols,
            "conversion_occurred": conversion
        })
        
    return pd.DataFrame(out)

In [121]:
# Configuration
file_path = "/mnt/c/GIS/git/AFOLU_GHG_flux_model/src/LULUCF/scripts/postprocessing/conversion_LUC_scenarios.xlsx"  # Update if needed
sheet_name = "scenarios"

lc_cols = [f"lc_{y}" for y in years]

In [122]:
# Load data
scenarios_df = pd.read_excel(file_path, sheet_name=sheet_name)

In [123]:
# Run classification
# Coerce scenarios_df to numeric
for c in lc_cols + ["driver", "sdpt", "logging_concession"]:
    if c in scenarios_df.columns: scenarios_df[c] = pd.to_numeric(scenarios_df[c], errors="coerce")
    
# Run classification
results_df = classify_dataframe(scenarios_df, rules, lc_cols)

# Output results 
print("\nClassification Results:")
print(results_df)

lu_vals: ['Grassland', 'Grassland', 'Grassland', 'Grassland', 'Grassland', 'Grassland', 'Grassland', 'Grassland', 'Grassland', 'Grassland']
tokens: ['G', 'G', 'G', 'G', 'G', 'G', 'G', 'G', 'G', 'G']
token_seq: GGGGGGGGGG
lu_vals: ['Grassland', 'Grassland', 'Grassland', 'Grassland', 'Grassland', 'Grassland', 'Grassland', 'Grassland', 'Grassland', 'Grassland']
tokens: ['G', 'G', 'G', 'G', 'G', 'G', 'G', 'G', 'G', 'G']
token_seq: GGGGGGGGGG
lu_vals: ['Grassland', 'Forest land', 'Grassland', 'Forest land', 'Grassland', 'Forest land', 'Grassland', 'Forest land', 'Grassland', 'Forest land']
tokens: ['G', 'F', 'G', 'F', 'G', 'F', 'G', 'F', 'G', 'F']
token_seq: GFGFGFGFGF
lu_vals: ['Forest land', 'Grassland', 'Forest land', 'Grassland', 'Forest land', 'Grassland', 'Forest land', 'Grassland', 'Forest land', 'Grassland']
tokens: ['F', 'G', 'F', 'G', 'F', 'G', 'F', 'G', 'F', 'G']
token_seq: FGFGFGFGFG
lu_vals: ['Forest land', 'Grassland', 'Grassland', 'Grassland', 'Forest land', 'Grassland', 'Gra

In [124]:
results_df.to_excel("/mnt/c/GIS/git/AFOLU_GHG_flux_model/src/LULUCF/scripts/postprocessing/conversion_LUC_scenario_results.xlsx", index=False)